# Coordinate transformation and CRS axis order

A coordinate pair is meaningful only with its coordinate reference system. This lab uses pyproj to transform points and makes the longitude/latitude versus x/y convention explicit.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'implementations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'implementations/python/crs-transform'))
from transform import transform_points


## 1. Geographic coordinates to Web Mercator

The input order is longitude, latitude because the wrapper sets always_xy. Web Mercator is useful for web-map display but is not equal-area and increasingly distorts scale toward the poles.

In [ ]:
points=np.array([[0,0],[10,50],[-73.9857,40.7484]])
projected=transform_points(points,'EPSG:4326','EPSG:3857')
for source,target in zip(points,projected):
    print(f'lon/lat {source} -> Web Mercator metres {target}')
restored=transform_points(projected,'EPSG:3857','EPSG:4326')
print('Maximum round-trip error (degrees):',np.max(np.abs(restored-points)))


## 2. UTM and the central meridian

Longitude 9°E is the central meridian of UTM zone 32N. At the equator, its easting is 500,000 metres by design.

In [ ]:
utm=transform_points([[9,0]],'EPSG:4326','EPSG:32632')
print('UTM zone 32N easting/northing:',utm[0])


## 3. Inspect the projection effect

The same three points have different coordinate values after projection. Their numeric magnitudes are not directly comparable across CRSs.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(points[:,0],points[:,1]); axes[0].set(title='Geographic coordinates',xlabel='Longitude (degrees)',ylabel='Latitude (degrees)')
axes[1].scatter(projected[:,0],projected[:,1]); axes[1].set(title='Web Mercator coordinates',xlabel='Easting (m)',ylabel='Northing (m)')
plt.tight_layout(); plt.show()


## Investigate

1. Why is the geographic origin also the Web Mercator origin?
2. Why does UTM zone 32N use a 500,000 m false easting at its central meridian?
3. What happens if longitude and latitude are swapped?
4. Why is Web Mercator unsuitable for area measurement at high latitudes?
5. What extra metadata is needed to reproduce a high-accuracy datum transformation?